# Overload margin with a scenario-level guarantee

GENCO's predicted line loadings are slightly off, so an overload screen needs a safety margin. Margins set from
individual lines are not a formal guarantee, because lines of one scenario are not independent. Here each scenario
contributes one number, the margin it needs so that none of its overloads is missed, and split-conformal calibration on
half of the N-2 scenarios gives:

> For a new N-2 scenario drawn like the calibration ones, the probability that it contains an overload we miss is at
> most 1 %.

The margin is then applied unchanged to N-5 to N-20 (an empirical check, not a guarantee) and compared with DC power flow
calibrated the same way. The calibration/test split is a seeded random half (`SEED = 0`), saved scenario by scenario in
`split_manifest.csv`. A last section repeats the calibration on 200 random splits.

Runtime: CPU, about 2 GB of downloads, 20 to 30 minutes. Output: `overload_margin.zip`.

In [ ]:
import importlib.util, subprocess, sys
need = {"huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "pandas": "pandas", "numpy": "numpy",
        "scipy": "scipy", "sklearn": "scikit-learn", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("installed now:", missing if missing else "nothing, all packages were already there")

## Settings

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
LOCAL_DIR = "genco-pf-contingency"
OUT_DIR = "overload_margin"

K_LEVELS = [2, 5, 10, 15, 20]             # outage levels to analyse
CALIB_K = 2                               # margins are calibrated on half of this level
N_PARTITIONS = {2: 20, 5: 5, 10: 5, 15: 5, 20: 5}   # blocks of 200 scenarios per level
MAX_DOWNLOAD_GB = 15
DOWNLOAD = True
CHUNK = 500                               # lower to 200 if memory runs out

ALPHA = 0.01                              # target: catch 99 % of overloads
FIT_TRUE = 0.80                           # residual exponent fitted on lines above 80 % true loading
CAL_TRUE_LINE_RULE = 0.95                     # line-level rule, kept for comparison
CAL_V = 0.01                              # voltage: near-limit rule (buses within 0.01 p.u. of a limit in truth)
KEEP_FLOOR = 0.3
WATCH_SHARE = 0.2                         # a line is "chronic" if flagged in >= 20 % of the calibration scenarios

NEWTON_LEVELS = [2, 10, 20]               # Newton frontier on these levels
N_NEWTON_RANDOM = 150                     # per level, random scenarios (used for all rates)
N_NEWTON_HARD = 60                        # per level, scenarios where GENCO misses an overload or voltage violation
MAX_STEPS = 10
TOL = 1e-8                                # convergence of a full AC solve (p.u.)
TOL_SCREEN = [1e-2, 1e-3]                 # adaptive rule: after one step, keep stepping while mismatch > this (p.u.)
BAD_VM = 0.01                             # a scenario is a "hard case" if GENCO's worst voltage error exceeds this (p.u.)
SEED = 0

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on"]:
        out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)


def conformal_margin(nc, alpha):
    # margin m such that, for exchangeable data, P(nonconformity <= m) >= 1 - alpha
    nc = np.sort(np.asarray(nc, float)[np.isfinite(nc)]); n = len(nc)
    if n == 0:
        return np.nan
    return float(nc[min(n, math.ceil((n + 1) * (1 - alpha))) - 1])


def evaluate(score, positive, sidx, n_scen, tau, scen_insecure):
    flag = score >= tau
    sent = np.zeros(n_scen, bool); sent[sidx[flag]] = True
    n_pos = int(positive.sum())
    return {"positives": n_pos, "caught (%)": 100 * (flag & positive).sum() / max(n_pos, 1),
            "missed": int((~flag & positive).sum()), "false alarms per scenario": (flag & ~positive).sum() / n_scen,
            "scenarios flagged (%)": 100 * sent.mean(),
            "secure scenarios cleared (%)": 100 * (~sent & ~scen_insecure).sum() / max((~scen_insecure).sum(), 1)}, flag


def qhigh(x, q):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return float(np.quantile(x, q, method="higher")) if len(x) else np.nan


def oracle_tau(score, positive, alpha):
    ps = np.sort(score[positive])[::-1]
    return ps[min(len(ps) - 1, math.ceil((1 - alpha) * len(ps)) - 1)] if len(ps) else np.nan


def test_view(T, k):
    m = ~T[k]["calib"].to_numpy()
    sub = T[k][m]
    S = SCEN[k][~SCEN[k]["calib"].to_numpy()]
    remap = np.full(len(SCEN[k]), -1); remap[S["sidx"].to_numpy()] = np.arange(len(S))
    return sub, S, remap[sub["sidx"].to_numpy()]

print("ready")

## 1. Download (released predictions and ground truth)

In [ ]:
RAW_PAT = re.compile(r"(?:^|/)(bus_data|branch_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
FLAT_PAT = re.compile(r"(?:^|/)(bus_data|branch_data)\.parquet$")
PART = 200

if DOWNLOAD:
    from huggingface_hub import HfApi, snapshot_download
    print("Listing the repository ...")
    all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True) if getattr(f, "size", None) is not None]
    local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
    stage1 = {p: s for p, s in all_files for k in K_LEVELS
              if p.startswith(f"mlflow/eval/test_{k}/") and (p.endswith("predictions.parquet") or p.endswith("meta.yaml"))
              and not re.search(r"_(gen|branch)_predictions\.parquet$", p)}
    new1 = sum(s for p, s in stage1.items() if not local_has(p)) / 1e9
    print(f"Stage 1 (predictions): {len(stage1)} files, {new1:.2f} GB not yet on disk")
    if new1 > MAX_DOWNLOAD_GB:
        raise RuntimeError("Stage 1 is above MAX_DOWNLOAD_GB")
    snapshot_download(REPO_ID, allow_patterns=list(stage1), local_dir=LOCAL_DIR)
    stage2, plan_rows = {}, []
    for k in K_LEVELS:
        pred_ids = scenario_ids(find_pred(LOCAL_DIR, k))
        counts = np.bincount(pred_ids // PART)
        raw = [(p, s, RAW_PAT.search(p)) for p, s in all_files if p.startswith(f"data/test_{k}/")]
        parts = sorted({int(m.group(2)) for _, _, m in raw if m})
        flat = [(p, s) for p, s in all_files if p.startswith(f"data/test_{k}/") and FLAT_PAT.search(p)]
        if parts:
            ranked = sorted(parts, key=lambda q: (-(counts[q] if q < len(counts) else 0), q))
            chosen = set(ranked[: N_PARTITIONS.get(k, 5)])
            files = {p: s for p, s, m in raw if m and int(m.group(2)) in chosen}
            n_pred = int(sum(counts[q] for q in chosen if q < len(counts)))
        else:
            files, chosen, n_pred = dict(flat), set(), len(pred_ids)
        stage2.update(files)
        plan_rows.append({"k": k, "blocks": len(chosen), "predicted scenarios in them": n_pred,
                          "GB (new)": round(sum(s for p, s in files.items() if not local_has(p)) / 1e9, 2)})
    display(pd.DataFrame(plan_rows))
    new2 = sum(s for p, s in stage2.items() if not local_has(p)) / 1e9
    print(f"Stage 2 (ground truth): {len(stage2)} files, {new2:.2f} GB not yet on disk")
    if new1 + new2 > MAX_DOWNLOAD_GB:
        raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB. Lower N_PARTITIONS or remove levels from K_LEVELS.")
    snapshot_download(REPO_ID, allow_patterns=list(stage2), local_dir=LOCAL_DIR)
    print("Download finished")
else:
    print("DOWNLOAD is False: using the files already in", LOCAL_DIR)

## 2. Line-level and bus-level tables 

In [ ]:
LINES, BUSES, SCEN, CHECKS, NR_STORE = {}, {}, {}, [], []
for k in K_LEVELS:
    bus_path, br_path, pred_path = find_local(LOCAL_DIR, k, "bus_data"), find_local(LOCAL_DIR, k, "branch_data"), find_pred(LOCAL_DIR, k)
    pred_names = ds.dataset(str(pred_path), format="parquet").schema.names
    colmap = {key: next((c for c in names if c in pred_names), None) for key, names in PRED_ALIASES.items()}
    ids_bus, ids_pred = scenario_ids(bus_path), scenario_ids(pred_path)
    common = np.intersect1d(ids_bus, ids_pred)
    if len(common) == 0:
        raise RuntimeError(f"k={k}: no scenario is in both the ground truth and the predictions")
    pred = read_cols(pred_path, ["scenario", "bus", colmap["vm"], colmap["va"]], scen=common)
    pred = pred.rename(columns={colmap["vm"]: "vm", colmap["va"]: "va"})
    pred["scenario"] = pred["scenario"].astype(np.int64); pred["bus"] = pred["bus"].astype(np.int64)
    pred = pred.drop_duplicates(["scenario", "bus"]).sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred_sc = pred["scenario"].to_numpy()
    do_nr = k in NEWTON_LEVELS
    rand_pick = set(RNG.choice(common, min(N_NEWTON_RANDOM, len(common)), replace=False).tolist()) if do_nr else set()
    n_hard = 0
    lines_parts, bus_parts, scen_parts = [], [], []
    for start in range(0, len(common), CHUNK):
        ids = common[start:start + CHUNK]
        lo, hi = np.searchsorted(pred_sc, ids[0], "left"), np.searchsorted(pred_sc, ids[-1], "right")
        A = load_chunk(bus_path, br_path, pred.iloc[lo:hi], ids)
        lines, buses, sc, chk = chunk_tables(A, near_line=KEEP_FLOOR)
        lines["sidx"] += start; buses["sidx"] += start; sc["sidx"] = start + np.arange(len(sc))
        lines_parts.append(lines); bus_parts.append(buses); scen_parts.append(sc)
        CHECKS.append({"k": k, **chk})
        if do_nr:
            hard = (sc["n_missed_overload_genco"].to_numpy() > 0) | (sc["n_missed_vviol_genco"].to_numpy() > 0)
            for i in range(len(ids)):
                if int(ids[i]) in rand_pick:
                    NR_STORE.append((k, "random", extract_scenario(A, i)))
                elif hard[i] and n_hard < N_NEWTON_HARD:
                    NR_STORE.append((k, "GENCO decision error", extract_scenario(A, i))); n_hard += 1
        del A
        print(f"  k={k}: {min(start + CHUNK, len(common)):,} / {len(common):,} scenarios", end="\r")
    LINES[k] = pd.concat(lines_parts, ignore_index=True)
    BUSES[k] = pd.concat(bus_parts, ignore_index=True)
    SCEN[k] = pd.concat(scen_parts, ignore_index=True)
    LINES[k]["res"] = SCEN[k]["res_p99"].to_numpy()[LINES[k]["sidx"].to_numpy()]
    BUSES[k]["res"] = SCEN[k]["res_p99"].to_numpy()[BUSES[k]["sidx"].to_numpy()]
    print(f"k={k:>2}: {len(common):,} scenarios | lines kept {len(LINES[k]):,} | buses kept {len(BUSES[k]):,}        ")
    del pred

calib_mask = np.zeros(len(SCEN[CALIB_K]), bool)
calib_mask[RNG.permutation(len(calib_mask))[: len(calib_mask) // 2]] = True
SCEN[CALIB_K]["calib"] = calib_mask
for k in K_LEVELS:
    if k != CALIB_K:
        SCEN[k]["calib"] = False
    for T in (LINES, BUSES):
        T[k]["calib"] = SCEN[k]["calib"].to_numpy()[T[k]["sidx"].to_numpy()]

chk = pd.DataFrame(CHECKS).groupby("k").agg(truth_mismatch_max_MW=("truth_mismatch_max_MW", "max"), va_unit=("va_unit", "first"))
chk["scenarios"] = [len(SCEN[k]) for k in chk.index]
chk["true overloaded lines"] = [int(SCEN[k].n_overload_true.sum()) for k in chk.index]
chk["true voltage violations"] = [int(SCEN[k].n_vviol_true.sum()) for k in chk.index]
chk.to_csv(Path(OUT_DIR) / "checks.csv")
display(chk)
print(f"Calibration set: {calib_mask.sum():,} scenarios of N-{CALIB_K}. Newton scenarios stored: {len(NR_STORE)}")

## 2b. The frozen calibration/test split (manifest for an audit)

In [ ]:
# Which scenarios set the margin (calibration half of N-2) and which test it (the other half; all of N-5 to N-20).
# The split is a seeded random half (SEED = 0, drawn in Section 2 before any margin is computed) and is never re-tuned.
import hashlib
rows = []
for k in K_LEVELS:
    S = SCEN[k]
    for sid, cal, n_o, n_v in zip(S["scenario"].to_numpy(), S["calib"].to_numpy(), S["n_overload_true"].to_numpy(), S["n_vviol_true"].to_numpy()):
        rows.append({"k": k, "scenario": int(sid), "role": "calibration" if cal else "test",
                     "true_overloads": int(n_o), "true_voltage_violations": int(n_v)})
MANIFEST = pd.DataFrame(rows).sort_values(["k", "scenario"]).reset_index(drop=True)
MANIFEST.to_csv(Path(OUT_DIR) / "split_manifest.csv", index=False)
digest = hashlib.sha256(MANIFEST.to_csv(index=False).encode()).hexdigest()
cal = MANIFEST[MANIFEST.role == "calibration"]
print(f"split manifest written: {len(MANIFEST):,} scenarios | calibration: {len(cal):,} N-{CALIB_K} scenarios, "
      f"{int((cal.true_overloads > 0).sum())} with an overload, {int((cal.true_voltage_violations > 0).sum())} with a voltage violation")
print("SHA-256 of the manifest (quote it in the write-up):", digest)
display(MANIFEST.groupby(["k", "role"]).size().unstack(fill_value=0))

## 3. Scenario-level scores

In [ ]:
from scipy.stats import beta as _beta

METHODS = {"GENCO": "ld_genco", "DC (active power)": "ld_dc_p", "DC (paper version)": "ld_dc_paper"}
NR_STORE.clear()      # Newton cases are not needed in this notebook


def calib_view(T, k):
    m = T[k]["calib"].to_numpy()
    sub = T[k][m]
    S = SCEN[k][SCEN[k]["calib"].to_numpy()]
    remap = np.full(len(SCEN[k]), -1); remap[S["sidx"].to_numpy()] = np.arange(len(S))
    return sub, S, remap[sub["sidx"].to_numpy()]


def element_scores(T, kind, col=None):
    """Score per line or bus: the margin it needs to be flagged (flagged iff score <= margin), and whether it is a
    true violation. Lines: 1 - predicted loading. Buses: predicted distance to the nearest voltage limit."""
    if kind == "line":
        s = 1.0 - T[col].to_numpy(np.float64)
        s = np.where(np.isfinite(s), s, np.inf)            # DC failed: no finite margin flags it
        pos = T.ld_true.to_numpy() >= 1.0
    else:
        vm, lo, hi = T.vm_genco.to_numpy(np.float64), T.vmin.to_numpy(np.float64), T.vmax.to_numpy(np.float64)
        s = np.minimum(vm - lo, hi - vm)
        vt = T.vm_true.to_numpy(np.float64)
        pos = (vt < lo) | (vt > hi)
    return s, pos


def scenario_scores(s, pos, sidx, n_scen):
    """Margin each scenario needs so that NONE of its true violations is missed (-inf if it has none)."""
    out = np.full(n_scen, -np.inf)
    np.maximum.at(out, sidx[pos], s[pos])
    return out


def conformal_q(scores, alpha):
    """Split-conformal quantile: for exchangeable scenarios, P(new score <= q) >= 1 - alpha."""
    x = np.sort(np.asarray(scores, float)); n = len(x)
    r = math.ceil((n + 1) * (1 - alpha))
    return float(x[r - 1]) if 1 <= r <= n else float("inf")


def cp_interval(k_, n, conf=0.95):
    a = 1 - conf
    lo = _beta.ppf(a / 2, k_, n - k_ + 1) if k_ > 0 else 0.0
    hi = _beta.ppf(1 - a / 2, k_ + 1, n - k_) if k_ < n else 1.0
    return lo, hi


def evaluate_margin(s, pos, sidx, n_scen, m):
    """Line/bus-level and scenario-level results of flagging every element with score <= m."""
    flag = s <= m
    sc = scenario_scores(s, pos, sidx, n_scen)
    has = sc > -np.inf                                   # scenarios with at least one true violation
    covered = sc[has] <= m
    lo, hi = cp_interval(int(covered.sum()), int(has.sum()))
    sent = np.zeros(n_scen, bool); sent[sidx[flag]] = True
    return {"scenarios with a violation": int(has.sum()),
            "of these, nothing missed (%)": 100 * covered.mean() if has.any() else np.nan,
            "95 % CI low (%)": 100 * lo, "95 % CI high (%)": 100 * hi,
            "all scenarios, nothing missed (%)": 100 * (1 - (~covered).sum() / n_scen),
            "violations caught (%)": 100 * (flag & pos).sum() / max(pos.sum(), 1),
            "false alarms per scenario": (flag & ~pos).sum() / n_scen,
            "scenarios flagged (%)": 100 * sent.mean()}


print("scenario-level tools ready")

## 4. Calibrate on half of N-2

In [ ]:
ALPHAS = [0.01, 0.05, 0.10]
RULES = {}
for name, col in METHODS.items():
    C, Sc, sc_idx = calib_view(LINES, CALIB_K)
    s, pos = element_scores(C, "line", col)
    S_cal = scenario_scores(s, pos, sc_idx, len(Sc))
    RULES[("overloads", name, "line-level", 0.01)] = conformal_q(s[pos & np.isfinite(s)], 0.01)
    for a in ALPHAS:
        RULES[("overloads", name, "scenario-level", a)] = conformal_q(S_cal[S_cal > -np.inf], a)
    if name == "GENCO":
        n_cal_over = int((S_cal > -np.inf).sum())
CB, Sb, sb_idx = calib_view(BUSES, CALIB_K)
s, pos = element_scores(CB, "bus")
S_calv = scenario_scores(s, pos, sb_idx, len(Sb))
RULES[("voltage", "GENCO", "line-level", 0.01)] = conformal_q(s[pos], 0.01)
for a in ALPHAS:
    RULES[("voltage", "GENCO", "scenario-level", a)] = conformal_q(S_calv[S_calv > -np.inf], a)
n_cal_v = int((S_calv > -np.inf).sum())
print(f"Calibration half of N-{CALIB_K}: {len(Sc)} scenarios, {n_cal_over} with at least one overload, {n_cal_v} with a voltage violation")
MARG = pd.DataFrame([{"violation": v, "method": mth, "rule": r, "alpha": a, "margin": m} for (v, mth, r, a), m in RULES.items()])
display(MARG)
print("Check: the line-level GENCO margin (expected 0.0445):", round(RULES[("overloads", "GENCO", "line-level", 0.01)], 4))
too_big = MARG[((MARG.violation == "overloads") & (MARG.margin > 1 - KEEP_FLOOR)) | ((MARG.violation == "voltage") & (MARG.margin > 0.08))]
if len(too_big):
    print("Note: some margins are larger than the stored tables cover, so their false alarms are undercounted:")
    display(too_big)

## 5. Test on the other half of N-2, and on N-5 to N-20 without re-tuning

In [ ]:
rows = []
for k in K_LEVELS:
    L, S, sidx = test_view(LINES, k)
    B, Sv, bidx = test_view(BUSES, k)
    for (viol, mth, rule, a), m in RULES.items():
        if viol == "overloads":
            s, pos = element_scores(L, "line", METHODS[mth]); res = evaluate_margin(s, pos, sidx, len(S), m)
        else:
            s, pos = element_scores(B, "bus"); res = evaluate_margin(s, pos, bidx, len(Sv), m)
        rows.append({"k": k, "violation": viol, "method": mth, "rule": rule, "alpha": a, "margin": m, **res})
RES = pd.DataFrame(rows)
RES.to_csv(Path(OUT_DIR) / "scenario_level_results.csv", index=False)
show = ["margin", "scenarios with a violation", "of these, nothing missed (%)", "95 % CI low (%)", "violations caught (%)",
        "false alarms per scenario", "scenarios flagged (%)"]
print("OVERLOADS (test scenarios only; N-2 = the half not used for calibration)")
display(RES[(RES.violation == "overloads") & (RES.alpha == 0.01)].set_index(["k", "method", "rule"])[show])
print("VOLTAGE VIOLATIONS (GENCO)")
display(RES[(RES.violation == "voltage") & (RES.alpha == 0.01)].set_index(["k", "rule"])[show])

## 6. Does the 99 % hold on N-2? 200 random calibration/test splits

In [ ]:
N_SPLITS = 200
Lk, Sk = LINES[CALIB_K], SCEN[CALIB_K]
n_s = len(Sk)
split_rows = []
for name in ["GENCO", "DC (active power)"]:
    s_all, pos_all = element_scores(Lk, "line", METHODS[name])
    sid_all = Lk["sidx"].to_numpy()
    S_all = scenario_scores(s_all, pos_all, sid_all, n_s)
    rng_s = np.random.default_rng(SEED + 7)
    for r in range(N_SPLITS):
        cal = np.zeros(n_s, bool); cal[rng_s.permutation(n_s)[: n_s // 2]] = True
        has = S_all > -np.inf
        m_sc = conformal_q(S_all[cal & has], 0.01)
        el_cal = cal[sid_all] & pos_all
        m_ln = conformal_q(s_all[el_cal], 0.01)
        te = ~cal & has
        for rule, m in [("scenario-level", m_sc), ("line-level", m_ln)]:
            split_rows.append({"method": name, "rule": rule, "split": r, "margin": m,
                               "of overloaded scenarios, nothing missed (%)": 100 * np.mean(S_all[te] <= m)})
SPL = pd.DataFrame(split_rows)
SPL.to_csv(Path(OUT_DIR) / "n2_random_splits.csv", index=False)
display(SPL.groupby(["method", "rule"])[["margin", "of overloaded scenarios, nothing missed (%)"]].describe(percentiles=[0.05, 0.5]).T)

## 7. Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
col = {"GENCO": "#1f5fa8", "DC (active power)": "#e07b20", "DC (paper version)": "#95a5a6"}
for (mth, rule), g in RES[(RES.violation == "overloads") & (RES.alpha == 0.01)].groupby(["method", "rule"]):
    if mth == "DC (paper version)":
        continue
    g = g.sort_values("k")
    ls = "-" if rule == "scenario-level" else ":"
    lab = f"{mth}, {rule}"
    y = g["of these, nothing missed (%)"]
    axes[0].errorbar(g.k + (0.15 if rule == "scenario-level" else -0.15), y, yerr=[y - g["95 % CI low (%)"], g["95 % CI high (%)"] - y],
                     ls=ls, marker="o", capsize=3, color=col[mth], label=lab)
    axes[1].plot(g.k, g["false alarms per scenario"], ls=ls, marker="o", color=col[mth], label=lab)
axes[0].axhline(99, color="k", lw=1, ls="--"); axes[0].text(K_LEVELS[-1] + 0.3, 99, "target 99 %", va="center", fontsize=8)
axes[0].set_ylabel("scenarios with an overload where none is missed (%)")
axes[0].set_title("Scenario-level guarantee (margins set once on N-2)")
axes[1].set_ylabel("false alarms per scenario"); axes[1].set_title("Price of the guarantee")
for ax in axes[:2]:
    ax.set_xticks(K_LEVELS); ax.set_xlabel("simultaneous outages (k)"); ax.grid(alpha=0.3)
axes[0].legend(frameon=False, fontsize=8, loc="lower left")
for (mth, rule), g in SPL[SPL.method == "GENCO"].groupby(["method", "rule"]):
    axes[2].hist(g["of overloaded scenarios, nothing missed (%)"], bins=25, alpha=0.6, label=rule)
axes[2].axvline(99, color="k", ls="--", lw=1)
axes[2].set_xlabel("scenarios with an overload where none is missed (%)"); axes[2].set_ylabel(f"of {N_SPLITS} random N-2 splits")
axes[2].set_title("GENCO on N-2: is the 99 % really 99 %?"); axes[2].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(Path(OUT_DIR) / "fig_scenario_level_margins.png", dpi=150); plt.show()

## Summary and output files

In [ ]:
def row(k, mth, rule, viol="overloads", a=0.01):
    r = RES[(RES.k == k) & (RES.method == mth) & (RES.rule == rule) & (RES.violation == viol) & (RES.alpha == a)]
    return r.iloc[0] if len(r) else None


for mth in ["GENCO", "DC (active power)"]:
    for rule in ["line-level", "scenario-level"]:
        parts = []
        for k in K_LEVELS:
            r = row(k, mth, rule)
            parts.append(f"N-{k}: {r['of these, nothing missed (%)']:.1f} % ({r['false alarms per scenario']:.2f} FA)")
        print(f"{mth:18s} {rule:24s} margin {row(K_LEVELS[0], mth, rule)['margin']:.4f} | " + " | ".join(parts))
summary = {"config": {"K_LEVELS": K_LEVELS, "CALIB_K": CALIB_K, "N_PARTITIONS": {str(k): v for k, v in N_PARTITIONS.items()},
                      "ALPHAS": ALPHAS, "N_SPLITS": N_SPLITS, "SEED": SEED},
           "calibration": {"scenarios": int(len(Sc)), "with_overload": n_cal_over, "with_voltage_violation": n_cal_v},
           "margins": MARG.to_dict("records"),
           "split_manifest": {"file": "split_manifest.csv", "sha256": digest, "scenarios": int(len(MANIFEST)),
                              "calibration_scenarios": int((MANIFEST.role == "calibration").sum())},
           "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__}}
(Path(OUT_DIR) / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
zip_path = Path(OUT_DIR).with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(OUT_DIR).iterdir()):
        zf.write(p, arcname=f"{Path(OUT_DIR).name}/{p.name}")
print(f"\nSaved {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB). Send this file back.")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass